## Topic: POST Requests 

### Agenda:
- 1. Introduction of POST Requests 

- 2. How FastAPI Processes a POST Request

- 3. POST Best Practices for ML APIs

- 4. Summary

###  1. Introduction of POST Requests 
-     ------HTTP request---------> 
- client <----------------------> server
-     <------ response ---------

- In this time, the client HTTP request type is POST.

- Step 1: Client Give an Post Request
    
    - POST carries data in the request body

- Step 2: Validation of the Pydantic Class
    - Pydantic models automatically parse and validate JSON bodies.

- Step 3: Store the Request data to database
    - if the data is validated, then store the data into database or json files as new records.
    - and add the status code.

    - else, using the status code we say data is not validated.


- request body: 
    - A request body is the portion of an HTTP request that contains data sent by the client to the server. It is typically used in HTTP methods such as POST, or PUT, to transmit structured data (eg. JSON, XML, from-data) for the purpose of creating or updating resources on the server.

    - The Server parses the request body to extract the necessary information and perform the intended operation.



In [ ]:
""" 
-> POST vs GET for ML — When to Use What:
=========================================

┌──────────────────────────────────────────────────────────────────┐
│                                                                  │
│ WRONG: Using GET for predictions                                 │
│  GET /predict?sepal_length=5.1&sepal_width=3.5&petal_length=1.4  │
│                                                                  │
│  Problems:                                                       │
│    • URL gets too long with many features                        │
│    • Sensitive data visible in URL (logs, browser history)       │
│    • Can't send complex nested data                              │
│    • Can't send binary data (images)                             │
│    • Browsers/proxies might cache results incorrectly            │
│                                                                  │
│ CORRECT: Using POST for predictions                              │
│  POST /predict                                                   │
│  Body: {"features": {"sepal_length": 5.1, ...}}                  │
│                                                                  │
│  Benefits:                                                       │
│    • No URL length limits                                        │
│    • Data hidden in body (not in logs)                           │
│    • Supports complex nested structures                          │
│    • Supports binary data (images, audio)                        │
│    • Semantically correct (you're submitting data)               │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘

"""

### 2. How FastAPI Processes a POST Request

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│              POST REQUEST LIFECYCLE IN FASTAPI                   │
│                                                                  │
│  Client sends:                                                   │
│  POST /predict                                                   │
│  Content-Type: application/json                                  │
│  {"sepal_length": 5.1, "sepal_width": 3.5,                       │
│   "petal_length": 1.4, "petal_width": 0.2}                       │
│                                                                  │
│  FastAPI processing pipeline:                                    │
│                                                                  │
│  1. ROUTING                                                      │
│     └── Match POST /predict → predict() function                 │
│                                                                  │
│  2. BODY PARSING                                                 │
│     └── Read request body                                        │
│     └── Parse JSON string → Python dict                          │
│                                                                  │
│  3. PYDANTIC VALIDATION                                          │
│     └── Create PredictionInput from dict                         │
│     └── Check: sepal_length is float?                            │
│     └── Check: sepal_length > 0?                                 │
│     └── Check: sepal_length <= 10?                               │
│     └── Check: all required fields present?                      │
│     └── If ANY check fails → return 422 error immediately        │
│                                                                  │
│  4. FUNCTION EXECUTION                                           │
│     └── Call predict(input_data) with validated data             │
│     └── Run your ML inference logic                              │
│                                                                  │
│  5. RESPONSE SERIALIZATION                                       │
│     └── Convert PredictionOutput → JSON                          │
│     └── Set Content-Type: application/json                       │
│     └── Set status code (200 by default)                         │
│                                                                  │
│  6. SEND RESPONSE                                                │
│     └── {"prediction": 0, "species": "setosa", "confidence": 0.97}│
│                                                                  │
└──────────────────────────────────────────────────────────────────┘


"""

### 3. POST Best Practices for ML APIs

In [ ]:
""" 
┌──────────────────────────────────────────────────────────────────┐
│            POST BEST PRACTICES FOR ML APIS                       │
│                                                                  │
│  1. ALWAYS use Pydantic schemas for request bodies               │
│     → Automatic validation, documentation, serialization         │
│                                                                  │
│  2. Validate inputs BEFORE they reach the model                  │
│     → Check ranges, NaN, Inf, missing features                   │
│     → A bad input can crash your model or give garbage output    │
│                                                                  │
│  3. Use appropriate status codes                                 │
│     → 200 for inference, 201 for creation, 202 for async         │
│     → 400 for bad features, 404 for missing model, 503 loading   │
│                                                                  │
│  4. Include request/response IDs for tracing                     │
│     → Essential for debugging in production                      │
│     → Link predictions to specific requests in logs              │
│                                                                  │
│  5. Return inference metadata                                    │
│     → Model version, latency, confidence                         │
│     → Helps with monitoring and debugging                        │
│                                                                  │
│  6. Support batch predictions                                    │
│     → 10-100x more efficient than individual calls               │
│     → Set reasonable max batch size                              │
│                                                                  │
│  7. Handle timeouts gracefully                                   │
│     → ML inference can be slow                                   │
│     → Return 504 if model takes too long                         │
│                                                                  │
│  8. Never expose internal details in errors                      │
│     → Don't leak stack traces, model paths, or DB info           │
│     → Return user-friendly error messages                        │
│                                                                  │
│  9. Log every prediction request                                 │
│     → Essential for monitoring, auditing, and retraining         │
│     → Include: input hash, output, latency, model version        │
│                                                                  │
│  10. Use response_model to control output                        │
│      → Never leak internal fields (passwords, hashes, paths)     │
│      → Keep responses clean and predictable                      │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘

"""

### 4. Summary

In [ ]:
""" 

┌──────────────────────────────────────────────────────────────────┐
│                  MODULE 06 — KEY TAKEAWAYS                       │
│                                                                  │
│  -  POST is the most important method for ML APIs                │
│  -  POST carries data in the request body (unlike GET)           │
│  -  Pydantic models automatically parse and validate JSON bodies │
│  -  Status codes: 200 (action), 201 (created), 202 (async)       │
│  -  Single prediction: POST /predict with feature dict           │
│  -  Batch prediction: POST /predict/batch with instance list     │
│  -  File uploads: UploadFile for images, audio, documents        │
│  -  Nested schemas: Complex real-world input structures          │
│  -  Custom validators: field_validator, model_validator          │
│  -  Error handling: 400, 404, 422, 503 with helpful messages     │
│  -  Response models: Filter output, hide sensitive data          │
│  -  Testing: TestClient with .post() and json= parameter         │
│  -  Always validate BEFORE inference (NaN, Inf, ranges)          │
│  -  Include tracking IDs and inference metadata                  │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘

"""